# 7. Describe Data Resources, Data Holders, and Research Fields

This notebook turns the final supplementary workbook into descriptive tables and figures for the paper. It deliberately **does not** change inclusion decisions, standardized names, display names, holder types, or holder countries. Those decisions are already final in the workbook.

The workflow is:

1. load the three analysis sheets from the supplementary workbook;
2. reshape semicolon-separated paper-level names into one mention per row;
3. count resources and holders overall;
4. connect resources to their holders and draw a treemap;
5. map selected ASJC subjects to five broad research fields;
6. count resources and holders within those fields; and
7. export distribution plots and a combined overview figure.

All counts use **unique papers** as the main measure. A source mentioned more than once in one paper therefore contributes only once.

## 1. Setup

This cell contains every path and display setting that you are likely to change. The supplementary workbook is the only analytical input. CSV tables are written to `processed_data`, while figures are written to `visualizations`.

In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from ast import literal_eval
from pathlib import Path
from textwrap import wrap
from xml.etree import ElementTree as ET
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# DATA_ROOT is configured through project_config.
SUPPLEMENTARY_FILE = DATA_ROOT / "interdependence_paper_supplementary_data_new.xlsx"
FIELD_MAP_FILE = DATA_ROOT / "asjc_subject_research_field_map.csv"
OUTPUT_DIR = DATA_ROOT / "processed_data"
FIGURE_DIR = DATA_ROOT / "visualizations"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TOP_N_OVERALL = 30
TOP_N_BY_FIELD = 10
TOP_N_HOLDERS_IN_TREEMAP = 15
# Hide resource labels below this share of the visible treemap area.
# The boxes and hover information remain available.
TREEMAP_MIN_RESOURCE_LABEL_SHARE = 0.004

RESOURCE = "data_resource_or_product"
HOLDER = "data_holder"

# Final holder categories and their consistent colors across all figures.
HOLDER_TYPE_COLORS = {
    "national public": "#2f6f9f",
    "international public": "#7a4f9a",
    "research": "#3f8f5f",
    "private": "#b84a4a",
    "ngo-led": "#c27c2c",
    "hybrid": "#8a8a8a",
    "not specified": "#bdbdbd",
}
HOLDER_TYPE_DISPLAY = {
    "national public": "National public",
    "international public": "International public",
    "research": "Research",
    "private": "Private",
    "ngo-led": "NGO-led",
    "hybrid": "Hybrid",
    "not specified": "Not specified",
}

print(f"Input workbook: {SUPPLEMENTARY_FILE}")
print(f"Table output folder: {OUTPUT_DIR}")
print(f"Figure output folder: {FIGURE_DIR}")

## 2. Small reusable functions

The final workbook already contains clean analytical labels, so only a few mechanical helpers remain:

- verify that required columns are present;
- split semicolon-separated lists;
- parse the ASJC list stored in each paper row;
- combine distinct metadata values; and
- save figures consistently.

There is no legacy holder-type recoding here. The notebook uses the final labels exactly as stored in the workbook.

In [ ]:
def require_columns(frame, required, sheet_name):
    """Stop with a clear message if the final workbook structure changes."""
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{sheet_name} is missing required columns: {missing}")


def clean_text(value):
    """Convert missing values to an empty string and remove extra whitespace."""
    if pd.isna(value):
        return ""
    return " ".join(str(value).replace("\xa0", " ").split())


def split_names(value):
    """Return unique, non-empty items from a semicolon-separated cell."""
    items = []
    seen = set()
    for item in clean_text(value).split(";"):
        item = clean_text(item)
        key = item.casefold()
        if item and key not in seen:
            items.append(item)
            seen.add(key)
    return items


def parse_subjects(value):
    """Parse ASJC subjects stored either as a Python-style list or semicolon text."""
    text = clean_text(value)
    if not text:
        return []
    try:
        parsed = literal_eval(text)
    except (ValueError, SyntaxError):
        parsed = text
    values = parsed if isinstance(parsed, (list, tuple, set)) else split_names(parsed)
    return list(dict.fromkeys(clean_text(value) for value in values if clean_text(value)))


def join_unique(values):
    """Join distinct metadata values while preserving their first-seen order."""
    combined = []
    seen = set()
    for value in values:
        for item in split_names(value):
            key = item.casefold()
            if key not in seen:
                combined.append(item)
                seen.add(key)
    return "; ".join(combined)


def first_text(values):
    """Return the first non-empty text value in a group."""
    for value in values:
        value = clean_text(value)
        if value:
            return value
    return ""


def save_matplotlib_figure(figure, filename_stem):
    """Save an editable SVG and a high-resolution PNG."""
    png_file = FIGURE_DIR / f"{filename_stem}.png"
    svg_file = FIGURE_DIR / f"{filename_stem}.svg"
    figure.savefig(png_file, dpi=300, bbox_inches="tight")
    figure.savefig(svg_file, bbox_inches="tight")
    print(f"Saved: {png_file}")
    print(f"Saved: {svg_file}")


def _has_zero_area_matrix(transform):
    """Return True for SVG matrix transforms that collapse an element to zero area."""
    transform = (transform or "").strip()
    if not (transform.startswith("matrix(") and transform.endswith(")")):
        return False
    try:
        values = [
            float(value)
            for value in transform[7:-1].replace(",", " ").split()
        ]
    except ValueError:
        return False
    return len(values) == 6 and all(abs(value) < 1e-12 for value in values[:4])


def save_plotly_figure_for_office(figure, filename_stem, width, height):
    """Export Plotly output as editable, Office-safe SVG and high-resolution PNG.

    Plotly can emit invisible empty paths and zero-area transforms. Browsers ignore
    them, but Microsoft Word's PDF exporter can drop the entire SVG. The canonical
    SVG written here removes only those non-rendering elements and keeps an untouched
    editable SVG alongside it.
    """
    editable_svg = FIGURE_DIR / f"{filename_stem}_editable.svg"
    office_svg = FIGURE_DIR / f"{filename_stem}.svg"
    png_file = FIGURE_DIR / f"{filename_stem}.png"

    figure.write_image(editable_svg, width=width, height=height)
    figure.write_image(png_file, width=width, height=height, scale=2)

    ET.register_namespace("", "http://www.w3.org/2000/svg")
    ET.register_namespace("xlink", "http://www.w3.org/1999/xlink")
    tree = ET.parse(editable_svg)
    root = tree.getroot()
    removed_empty_paths = 0
    removed_zero_transforms = 0

    for parent in list(root.iter()):
        for child in list(parent):
            local_name = child.tag.rsplit("}", 1)[-1]
            if local_name == "path" and not child.get("d", "").strip():
                parent.remove(child)
                removed_empty_paths += 1
            elif _has_zero_area_matrix(child.get("transform")):
                parent.remove(child)
                removed_zero_transforms += 1

    # Use an explicit SVG 1.1 canvas and avoid root-level clipping differences.
    root.set("version", "1.1")
    root.set("width", str(width))
    root.set("height", str(height))
    root.set("viewBox", f"0 0 {width} {height}")
    root.attrib.pop("overflow", None)
    tree.write(office_svg, encoding="utf-8", xml_declaration=True)

    # Reparse and fail loudly if the export is not well-formed.
    ET.parse(office_svg)
    print(f"Saved editable SVG: {editable_svg}")
    print(f"Saved Office-compatible SVG: {office_svg}")
    print(f"Saved high-resolution PNG: {png_file}")
    print(
        "Removed non-rendering SVG elements: "
        f"{removed_empty_paths} empty paths, "
        f"{removed_zero_transforms} zero-area transforms"
    )

## 3. Load only the final workbook columns used below

The workbook contains additional documentation sheets, but this analysis needs only:

- `paper_source_data`: one row per paper, including final resource and holder lists;
- `standardized_sources`: the relationship between resources and holders; and
- `visualization_names`: final figure labels and reviewed holder metadata.

Reading a defined subset of columns makes the notebook faster and makes its dependencies explicit.

In [ ]:
PAPER_COLUMNS = [
    "doi", "publication_year", "title", "affiliations", "affiliations_countries",
    "asjc_subjects", "standardized_data_resource_names",
    "standardized_data_holder_names", "used_in_analysis",
]
SOURCE_COLUMNS = [
    "standardized_data_resource_name", "standardized_data_holder_name",
    "standardized_data_holder_type", "standardized_data_holder_country_ISO2", "URL",
]
VISUALIZATION_COLUMNS = [
    "standardized_data_resource_name", "data_resource_for_visualization",
    "standardized_data_holder_name", "data_holder_for_visualization",
    "standardized_data_holder_type", "standardized_data_holder_country_ISO2",
]

papers = pd.read_excel(
    SUPPLEMENTARY_FILE, sheet_name="paper_source_data", usecols=PAPER_COLUMNS
)
source_metadata_raw = pd.read_excel(
    SUPPLEMENTARY_FILE, sheet_name="standardized_sources", usecols=SOURCE_COLUMNS
)
visualization_metadata = pd.read_excel(
    SUPPLEMENTARY_FILE, sheet_name="visualization_names", usecols=VISUALIZATION_COLUMNS
)

require_columns(papers, PAPER_COLUMNS, "paper_source_data")
require_columns(source_metadata_raw, SOURCE_COLUMNS, "standardized_sources")
require_columns(visualization_metadata, VISUALIZATION_COLUMNS, "visualization_names")

# The row number is a stable paper identifier within this workbook.
papers.insert(0, "paper_id", pd.RangeIndex(1, len(papers) + 1))
papers["used_in_analysis"] = papers["used_in_analysis"].fillna(False).astype(bool)

# Standardized names are populated for retained papers. Keeping the explicit flag
# makes the inclusion rule visible and prevents accidental use of excluded rows.
analysis_papers = papers.loc[papers["used_in_analysis"]].copy()

print(f"All paper rows: {len(papers):,}")
print(f"Rows marked for analysis: {len(analysis_papers):,}")
print(f"Standardized-source metadata rows: {len(source_metadata_raw):,}")
print(f"Visualization metadata rows: {len(visualization_metadata):,}")

## 4. Build clean metadata tables

The supplementary workbook separates canonical names from figure-friendly names. This section creates two compact lookup tables:

- one row per data resource/product; and
- one row per data holder.

The canonical names remain the keys used for counting. Display names are used only in figures.

In [ ]:
for frame in (source_metadata_raw, visualization_metadata):
    for column in frame.columns:
        frame[column] = frame[column].map(clean_text)

resource_display_lookup = (
    visualization_metadata.loc[
        visualization_metadata["standardized_data_resource_name"].ne("")
    ]
    .groupby("standardized_data_resource_name")["data_resource_for_visualization"]
    .agg(first_text)
    .to_dict()
)

holder_display_metadata = (
    visualization_metadata.loc[
        visualization_metadata["standardized_data_holder_name"].ne("")
    ]
    .groupby("standardized_data_holder_name", as_index=False)
    .agg(
        data_holder_for_visualization=("data_holder_for_visualization", first_text),
        standardized_data_holder_type=("standardized_data_holder_type", first_text),
        standardized_data_holder_country_ISO2=("standardized_data_holder_country_ISO2", first_text),
    )
)

# Use standardized_sources as a metadata fallback for holders that have no
# dedicated visualization_names row. Visualization metadata takes precedence.
holder_fallback_metadata = source_metadata_raw[[
    "standardized_data_holder_name", "standardized_data_holder_type",
    "standardized_data_holder_country_ISO2",
]].copy()
holder_fallback_metadata["standardized_data_holder_name"] = holder_fallback_metadata[
    "standardized_data_holder_name"
].map(split_names)
holder_fallback_metadata = holder_fallback_metadata.explode(
    "standardized_data_holder_name"
).dropna(subset=["standardized_data_holder_name"])
holder_fallback_metadata = (
    holder_fallback_metadata.groupby("standardized_data_holder_name", as_index=False)
    .agg(
        fallback_holder_type=("standardized_data_holder_type", first_text),
        fallback_holder_country=("standardized_data_holder_country_ISO2", first_text),
    )
)

holder_metadata = holder_fallback_metadata.merge(
    holder_display_metadata, on="standardized_data_holder_name", how="outer"
)
holder_metadata["standardized_data_holder_type"] = holder_metadata[
    "standardized_data_holder_type"
].mask(
    holder_metadata["standardized_data_holder_type"].fillna("").eq(""),
    holder_metadata["fallback_holder_type"],
)
holder_metadata["standardized_data_holder_country_ISO2"] = holder_metadata[
    "standardized_data_holder_country_ISO2"
].mask(
    holder_metadata["standardized_data_holder_country_ISO2"].fillna("").eq(""),
    holder_metadata["fallback_holder_country"],
)
holder_metadata = holder_metadata.drop(
    columns=["fallback_holder_type", "fallback_holder_country"]
)
holder_metadata["data_holder_for_visualization"] = holder_metadata[
    "data_holder_for_visualization"
].mask(
    holder_metadata["data_holder_for_visualization"].fillna("").eq(""),
    holder_metadata["standardized_data_holder_name"],
)

resource_metadata = (
    source_metadata_raw.loc[
        source_metadata_raw["standardized_data_resource_name"].ne("")
    ]
    .groupby("standardized_data_resource_name", as_index=False)
    .agg(
        standardized_data_holder_name=("standardized_data_holder_name", join_unique),
        standardized_data_holder_type=("standardized_data_holder_type", join_unique),
        standardized_data_holder_country_ISO2=("standardized_data_holder_country_ISO2", join_unique),
        URL=("URL", join_unique),
    )
)
resource_metadata["data_resource_for_visualization"] = resource_metadata[
    "standardized_data_resource_name"
].map(resource_display_lookup)
resource_metadata["data_resource_for_visualization"] = resource_metadata[
    "data_resource_for_visualization"
].fillna(resource_metadata["standardized_data_resource_name"])

print(f"Unique resources with metadata: {len(resource_metadata):,}")
print(f"Unique holders with display metadata: {len(holder_metadata):,}")
display(resource_metadata.head())
display(holder_metadata.head())

## 5. Reshape paper-level lists into mention-level tables

The paper sheet stores several standardized names in one semicolon-separated cell. Counting is clearer after reshaping:

- `resource_mentions` has one paper–resource pair per row;
- `holder_mentions` has one deduplicated paper–holder pair per row, combining the paper-level standardized holder field with holders explicitly inferred through the reviewed resource–holder dictionary; and
- `source_mentions` combines both tables for shared counting functions.

Duplicate paper–name pairs are removed before any counts are calculated. A paper linked to the same holder through both routes therefore contributes only once to that holder.

In [ ]:
PAPER_CONTEXT = [
    "paper_id", "doi", "publication_year", "title", "affiliations",
    "affiliations_countries", "asjc_subjects",
]

resource_mentions = analysis_papers[PAPER_CONTEXT + ["standardized_data_resource_names"]].copy()
resource_mentions["standardized_name"] = resource_mentions[
    "standardized_data_resource_names"
].map(split_names)
resource_mentions = resource_mentions.explode("standardized_name").dropna(subset=["standardized_name"])
resource_mentions = resource_mentions.drop(columns="standardized_data_resource_names")
resource_mentions = resource_mentions.drop_duplicates(["paper_id", "standardized_name"])
resource_mentions["entity_type"] = RESOURCE
resource_mentions = resource_mentions.merge(
    resource_metadata,
    left_on="standardized_name",
    right_on="standardized_data_resource_name",
    how="left",
)

# Build the reviewed resource-to-holder dictionary once. A resource can map
# to more than one holder, so holder names are exploded before the join.
resource_holder_map = source_metadata_raw[
    ["standardized_data_resource_name", "standardized_data_holder_name"]
].copy()
resource_holder_map["standardized_data_holder_name"] = resource_holder_map[
    "standardized_data_holder_name"
].map(split_names)
resource_holder_map = (
    resource_holder_map.explode("standardized_data_holder_name")
    .dropna(subset=["standardized_data_holder_name"])
    .drop_duplicates()
)

# Route 1: holders already stored in the paper-level standardized holder field.
# Notebook 6.1 currently populates this field from both holder and resource rows.
paper_field_holder_pairs = analysis_papers[["paper_id", "standardized_data_holder_names"]].copy()
paper_field_holder_pairs["standardized_name"] = paper_field_holder_pairs[
    "standardized_data_holder_names"
].map(split_names)
paper_field_holder_pairs = (
    paper_field_holder_pairs.explode("standardized_name")
    .dropna(subset=["standardized_name"])
    [["paper_id", "standardized_name"]]
    .drop_duplicates()
    .assign(holder_link_route="paper-level holder field")
)

# Route 2: holders inferred from every mentioned resource via the reviewed dictionary.
resource_inferred_holder_pairs = (
    resource_mentions[["paper_id", "standardized_name"]]
    .merge(
        resource_holder_map,
        left_on="standardized_name",
        right_on="standardized_data_resource_name",
        how="inner",
    )
    [["paper_id", "standardized_data_holder_name"]]
    .rename(columns={"standardized_data_holder_name": "standardized_name"})
    .drop_duplicates()
    .assign(holder_link_route="resource-to-holder mapping")
)

# The analytical holder table is the union of both routes. Grouping before
# counting ensures one paper contributes at most once to the same holder.
holder_pair_routes = pd.concat(
    [paper_field_holder_pairs, resource_inferred_holder_pairs], ignore_index=True
).drop_duplicates(["paper_id", "standardized_name", "holder_link_route"])
holder_mentions = (
    holder_pair_routes.groupby(["paper_id", "standardized_name"], as_index=False)
    .agg(holder_link_routes=("holder_link_route", lambda values: "; ".join(sorted(set(values)))))
    .merge(analysis_papers[PAPER_CONTEXT], on="paper_id", how="left")
)
holder_mentions["entity_type"] = HOLDER
holder_mentions = holder_mentions.merge(
    holder_metadata,
    left_on="standardized_name",
    right_on="standardized_data_holder_name",
    how="left",
)

source_mentions = pd.concat(
    [
        resource_mentions[PAPER_CONTEXT + ["standardized_name", "entity_type"]],
        holder_mentions[PAPER_CONTEXT + ["standardized_name", "entity_type"]],
    ],
    ignore_index=True,
)

print(f"Unique paper–resource pairs: {len(resource_mentions):,}")
print(f"Paper-level holder-field pairs: {len(paper_field_holder_pairs):,}")
print(f"Resource-inferred paper–holder pairs: {len(resource_inferred_holder_pairs):,}")
print(f"Combined unique paper–holder pairs: {len(holder_mentions):,}")
print(f"Papers with at least one retained source: {source_mentions['paper_id'].nunique():,}")
display(source_mentions.head())

In [ ]:
len(source_mentions['paper_id'].unique())

## 6. Overall resource and holder counts

For each canonical name, the tables report:

- `source_mentions`: number of unique paper–source pairs; and
- `unique_papers`: number of different papers containing the source.

These values are normally identical because duplicate paper–source pairs were removed above. Holder counts union the paper-level standardized holder field with explicit resource-inferred holder links, then deduplicate at the paper–holder level. The upstream paper field currently already contains the inferred links; rebuilding the union here makes that rule visible and testable without inflating counts. Both count columns are retained to make the counting unit explicit and to keep the output extensible.

In [ ]:
def count_sources(frame, entity_type):
    """Rank canonical names within one entity type."""
    counts = (
        frame.loc[frame["entity_type"].eq(entity_type)]
        .groupby("standardized_name", as_index=False)
        .agg(
            source_mentions=("standardized_name", "size"),
            unique_papers=("paper_id", "nunique"),
        )
        .sort_values(
            ["unique_papers", "source_mentions", "standardized_name"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )
    counts.insert(0, "rank", np.arange(1, len(counts) + 1))
    counts.insert(1, "entity_type", entity_type)
    return counts


resource_counts = count_sources(source_mentions, RESOURCE)
holder_counts = count_sources(source_mentions, HOLDER)
analysis_paper_count = analysis_papers["paper_id"].nunique()
holder_counts["share_of_analysis_papers"] = (
    holder_counts["unique_papers"] / analysis_paper_count
)
holder_paper_shares = (
    holder_counts.merge(
        holder_metadata[["standardized_data_holder_name", "data_holder_for_visualization"]],
        left_on="standardized_name",
        right_on="standardized_data_holder_name",
        how="left",
    )
    [["data_holder_for_visualization", "unique_papers", "share_of_analysis_papers"]]
    .rename(columns={
        "data_holder_for_visualization": "data_holder",
        "unique_papers": "paper_count",
    })
)

resource_counts.to_csv(
    OUTPUT_DIR / "top_data_resource_product_counts.csv", index=False, encoding="utf-8-sig"
)
holder_counts.to_csv(
    OUTPUT_DIR / "top_data_holder_counts.csv", index=False, encoding="utf-8-sig"
)

display(resource_counts.head(TOP_N_OVERALL))
display(holder_counts.head(TOP_N_OVERALL))
display(holder_paper_shares)

### Plot the overall rankings

The bars use unique-paper counts and canonical names. Horizontal bars keep long source names readable.

In [ ]:
def plot_top_counts(counts, title, filename_stem, color):
    plot_data = counts.head(TOP_N_OVERALL).sort_values("unique_papers").copy()
    plot_data["label"] = plot_data["standardized_name"].map(
        lambda value: "\n".join(wrap(value, width=34))
    )
    height = max(6, 0.33 * len(plot_data) + 1.5)
    figure, axis = plt.subplots(figsize=(10, height))
    bars = axis.barh(plot_data["label"], plot_data["unique_papers"], color=color)
    if "share_of_analysis_papers" in plot_data.columns:
        axis.bar_label(
            bars,
            labels=[f"{value:.1%}" for value in plot_data["share_of_analysis_papers"]],
            padding=3,
            fontsize=8,
        )
        axis.margins(x=0.12)
    axis.set(title=title, xlabel="Unique papers", ylabel="")
    axis.grid(axis="x", alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)
    figure.tight_layout()
    save_matplotlib_figure(figure, filename_stem)
    plt.show()


plot_top_counts(
    resource_counts,
    f"Top {TOP_N_OVERALL} data resources/products",
    "top_data_resource_product_counts",
    "#2f6f9f",
)
plot_top_counts(
    holder_counts,
    f"Top {TOP_N_OVERALL} data holders",
    "top_data_holder_counts",
    "#7a4f9a",
)

## 7. Resources nested within data holders

The overall resource list does not show who provides each resource. This section joins paper–resource mentions to the reviewed resource–holder relationships in `standardized_sources`.

One resource may be linked to more than one holder. In that case, the paper–resource pair appears once under each reviewed holder in the treemap table. The treemap is therefore a view of **resource–holder relationships**, not a mutually exclusive partition of papers.

Holder colors are explained in the holder-type pie chart, so the treemap has no separate legend. Each holder label reports the number of retained analysis papers that relied on it; each paper is counted at most once per holder. Quoted residual boxes such as `“NCBI data”` represent paper–holder links that were present in the standardized holder field but could not be attributed to a specific mapped resource. Resource and residual labels are shown only when their boxes exceed the readable-area cutoff defined in setup; smaller boxes remain visible and retain their hover information.

In [ ]:
# Add display metadata to the reviewed resource–holder relationships built above.
resource_holder_map_with_metadata = resource_holder_map.merge(
    holder_metadata, on="standardized_data_holder_name", how="left"
)

treemap_mentions = resource_mentions[
    ["paper_id", "standardized_name", "data_resource_for_visualization"]
].merge(
    resource_holder_map_with_metadata,
    left_on="standardized_name",
    right_on="standardized_data_resource_name",
    how="left",
)

treemap_mentions["standardized_data_holder_name"] = treemap_mentions[
    "standardized_data_holder_name"
].fillna("Data holder not specified")
treemap_mentions["data_holder_for_visualization"] = treemap_mentions[
    "data_holder_for_visualization"
].fillna(treemap_mentions["standardized_data_holder_name"])
treemap_mentions["standardized_data_holder_type"] = treemap_mentions[
    "standardized_data_holder_type"
].fillna("not specified")
treemap_mentions["data_resource_for_visualization"] = treemap_mentions[
    "data_resource_for_visualization"
].fillna(treemap_mentions["standardized_name"])

holder_resource_counts = (
    treemap_mentions.groupby(
        [
            "standardized_data_holder_name", "data_holder_for_visualization",
            "standardized_data_holder_type", "standardized_name",
            "data_resource_for_visualization",
        ],
        as_index=False,
    )
    .agg(unique_papers=("paper_id", "nunique"))
)

# Identify holder links that cannot be attributed to any mapped resource.
# These are the paper-level holder pairs left after removing every pair that
# can also be reconstructed through the resource-to-holder dictionary.
holder_only_pairs = (
    paper_field_holder_pairs.merge(
        resource_inferred_holder_pairs[["paper_id", "standardized_name"]],
        on=["paper_id", "standardized_name"],
        how="left",
        indicator=True,
    )
    .loc[lambda frame: frame["_merge"].eq("left_only")]
    .drop(columns="_merge")
)

# Turn those holder-only links into residual treemap boxes. Quotation marks
# signal that labels such as “NCBI data” reflect textual ambiguity rather
# than a specifically identified data resource.
holder_only_counts = (
    holder_only_pairs.merge(
        holder_metadata,
        left_on="standardized_name",
        right_on="standardized_data_holder_name",
        how="left",
    )
    .assign(
        data_holder_for_visualization=lambda frame: frame[
            "data_holder_for_visualization"
        ].fillna(frame["standardized_name"]),
        standardized_data_holder_type=lambda frame: frame[
            "standardized_data_holder_type"
        ].fillna("not specified"),
    )
    .groupby(
        ["standardized_name", "data_holder_for_visualization",
         "standardized_data_holder_type"],
        as_index=False,
    )
    .agg(unique_papers=("paper_id", "nunique"))
    .rename(columns={"standardized_name": "standardized_data_holder_name"})
)
holder_only_counts["standardized_name"] = (
    holder_only_counts["standardized_data_holder_name"]
    + " (holder-only textual mention)"
)
holder_only_counts["data_resource_for_visualization"] = (
    "“" + holder_only_counts["data_holder_for_visualization"] + " data”"
)

holder_totals = (
    holder_resource_counts.groupby("standardized_data_holder_name", as_index=False)
    .agg(holder_total_resource_papers=("unique_papers", "sum"))
    .sort_values("holder_total_resource_papers", ascending=False)
)
top_holders = holder_totals.head(TOP_N_HOLDERS_IN_TREEMAP)["standardized_data_holder_name"]
treemap_counts = pd.concat(
    [
        holder_resource_counts.loc[
            holder_resource_counts["standardized_data_holder_name"].isin(top_holders)
        ],
        holder_only_counts.loc[
            holder_only_counts["standardized_data_holder_name"].isin(top_holders)
        ],
    ],
    ignore_index=True,
)
treemap_counts = treemap_counts.merge(holder_totals, on="standardized_data_holder_name")
treemap_counts.to_csv(
    OUTPUT_DIR / "data_holder_resource_treemap_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

treemap = px.treemap(
    treemap_counts,
    path=["data_holder_for_visualization", "data_resource_for_visualization"],
    values="unique_papers",
    color="standardized_data_holder_type",
    color_discrete_map=HOLDER_TYPE_COLORS,
    hover_data={
        "standardized_data_holder_name": True,
        "standardized_name": True,
        "holder_total_resource_papers": ":,",
    },
    title=f"Resources within the top {TOP_N_HOLDERS_IN_TREEMAP} data holders",
)

# Holder labels use the combined direct-plus-inferred paper–holder table.
# Each paper counts once per holder and the denominator is every analysis paper.
treemap_holder_paper_shares = (
    holder_mentions.loc[holder_mentions["standardized_name"].isin(top_holders)]
    .assign(data_holder_for_visualization=lambda frame: frame[
        "data_holder_for_visualization"
    ].fillna(frame["standardized_name"]))
    .groupby("data_holder_for_visualization", as_index=False)
    .agg(paper_count=("paper_id", "nunique"))
    .sort_values("paper_count", ascending=False)
)
treemap_holder_paper_shares["share_of_analysis_papers"] = (
    treemap_holder_paper_shares["paper_count"] / analysis_paper_count
)
treemap_holder_stats = treemap_holder_paper_shares.set_index(
    "data_holder_for_visualization"
)[["paper_count", "share_of_analysis_papers"]].to_dict(orient="index")

# Use a larger two-line holder header without percentages. Resource labels
# retain their smaller font through inline styling.
visible_treemap_total = treemap_counts["unique_papers"].sum()
treemap_labels = []
treemap_text = []
for label, parent, value in zip(
    treemap.data[0].labels, treemap.data[0].parents, treemap.data[0].values
):
    if parent == "":
        holder_stats = treemap_holder_stats[label]
        holder_label = (
            f"<span style='font-size:16px'><b>{label}</b></span> · "
            f"<span style='font-size:15px'>"
            f"{holder_stats['paper_count']:,.0f} papers</span>"
        )
        treemap_labels.append(holder_label)
        treemap_text.append(holder_label)
    else:
        treemap_labels.append(label)
        area_share = value / visible_treemap_total if visible_treemap_total else 0
        treemap_text.append(
            f"<span style='font-size:13px'>{label}<br>{value:,.0f} papers</span>"
            if area_share >= TREEMAP_MIN_RESOURCE_LABEL_SHARE else ""
        )

treemap.update_traces(
    labels=treemap_labels,
    text=treemap_text,
    texttemplate="%{text}",
    textinfo="text",
    # The scalar base size controls the parent-header height and renders
    # reliably; resource labels override it with the 13 px span above.
    textfont=dict(size=17),
)
treemap.update_layout(
    height=900,
    width=1500,
    margin=dict(t=60, l=10, r=10, b=10),
    showlegend=False,
)
treemap.write_html(FIGURE_DIR / "data_holder_resource_treemap.html")
treemap.write_image(FIGURE_DIR / "data_holder_resource_treemap.svg", width=1500, height=900)
treemap.show()

display(treemap_holder_paper_shares)
display(treemap_counts.head(30))

In [ ]:
27020/86368

## 8. Map ASJC subjects to broad research fields

The supplementary workbook stores detailed ASJC subjects. For interpretation, this analysis maps the originally selected subjects into five broad fields.

Two versions are retained:

- **all field memberships**: a paper can contribute to several broad fields;
- **primary field**: each paper contributes to one field, chosen by the largest number of matched ASJC subjects. Ties follow the order shown in `RESEARCH_FIELD_CLUSTERS`.

The mapping below is the original analytical classification used in this notebook. It is kept visible so the selected subjects and grouping remain directly reviewable.

In [ ]:
RESEARCH_FIELD_CLUSTERS = {
    "Ecology, Biodiversity & Conservation": [
        "Animal Science and Zoology", "Conservation", "Ecology",
        "Ecology, Evolution, Behavior and Systematics", "Nature and Landscape Conservation",
    ],
    "Earth, Climate, Ocean & Water": [
        "Atmospheric Science", "Earth and Planetary Sciences (miscellaneous)",
        "Earth-Surface Processes", "Geography, Planning and Development", "Geology",
        "Geochemistry and Petrology", "Global and Planetary Change", "Oceanography",
        "Paleontology", "Soil Science", "Water Science and Technology",
    ],
    "Agriculture, Forestry & Food": [
        "Agronomy and Crop Science", "Food Science", "Forestry", "Horticulture", "Plant Science",
    ],
    "Environmental Health, Pollution & Risk": [
        "Environmental Chemistry", "Environmental Engineering", "Health, Toxicology and Mutagenesis",
        "Pollution", "Toxicology", "Waste Management and Disposal",
    ],
    "Social, Economic & Policy": [
        "Development", "Economics and Econometrics", "Law",
        "Management, Monitoring, Policy and Law", "Political Science and International Relations",
        "Public Administration", "Sociology and Political Science", "Urban Studies",
    ],
}
FIELD_ORDER = list(RESEARCH_FIELD_CLUSTERS)
FIELD_DISPLAY = {
    "Ecology, Biodiversity & Conservation": "Ecology & Conservation",
}

subject_rows = analysis_papers[["paper_id", "asjc_subjects"]].copy()
subject_rows["asjc_subject"] = subject_rows["asjc_subjects"].map(parse_subjects)
subject_rows = subject_rows.explode("asjc_subject").dropna(subset=["asjc_subject"])
subject_rows = subject_rows[["paper_id", "asjc_subject"]].drop_duplicates()

field_map = pd.DataFrame(
    [
        {"research_field": field, "asjc_subject": subject}
        for field, subjects in RESEARCH_FIELD_CLUSTERS.items()
        for subject in subjects
    ]
)

subject_mapping_audit = (
    subject_rows.groupby("asjc_subject", as_index=False)
    .agg(papers=("paper_id", "nunique"))
    .merge(field_map, on="asjc_subject", how="left")
    .assign(research_field=lambda frame: frame["research_field"].fillna("Unassigned"))
    .sort_values(["research_field", "papers", "asjc_subject"], ascending=[True, False, True])
)
subject_mapping_audit.to_csv(
    OUTPUT_DIR / "asjc_subject_research_field_audit.csv", index=False, encoding="utf-8-sig"
)
print(f"Observed ASJC subjects: {subject_rows['asjc_subject'].nunique():,}")
print(f"Subjects included in the original field classification: {len(field_map):,}")
paper_field_subjects = subject_rows.merge(field_map, on="asjc_subject", how="inner")

all_field_memberships = paper_field_subjects[
    ["paper_id", "research_field"]
].drop_duplicates()

field_order = {field: position for position, field in enumerate(FIELD_ORDER)}
primary_field_memberships = (
    paper_field_subjects.groupby(["paper_id", "research_field"], as_index=False)
    .agg(matched_subjects=("asjc_subject", "nunique"))
    .assign(field_order=lambda frame: frame["research_field"].map(field_order))
    .sort_values(
        ["paper_id", "matched_subjects", "field_order"],
        ascending=[True, False, True],
    )
    .drop_duplicates("paper_id")
    [["paper_id", "research_field"]]
)

field_paper_counts = (
    all_field_memberships.groupby("research_field", as_index=False)
    .agg(papers_in_field=("paper_id", "nunique"))
)
primary_field_paper_counts = (
    primary_field_memberships.groupby("research_field", as_index=False)
    .agg(papers_in_primary_field=("paper_id", "nunique"))
)

field_paper_counts.to_csv(
    OUTPUT_DIR / "research_field_paper_counts.csv", index=False, encoding="utf-8-sig"
)
primary_field_paper_counts.to_csv(
    OUTPUT_DIR / "primary_research_field_paper_counts.csv", index=False, encoding="utf-8-sig"
)

print(f"Papers mapped to at least one broad field: {all_field_memberships['paper_id'].nunique():,}")
display(field_paper_counts)
display(primary_field_paper_counts)

## 9. Count resources and holders within research fields

The same counting rule is applied after joining source mentions to field memberships. The rank restarts within each field.

In [ ]:
def count_sources_by_field(source_rows, memberships, entity_type, rank_column):
    """Count and rank one source type within each broad research field."""
    joined = source_rows.loc[source_rows["entity_type"].eq(entity_type)].merge(
        memberships, on="paper_id", how="inner"
    )
    counts = (
        joined.groupby(["research_field", "standardized_name"], as_index=False)
        .agg(
            source_mentions=("standardized_name", "size"),
            unique_papers=("paper_id", "nunique"),
        )
        .sort_values(
            ["research_field", "unique_papers", "source_mentions", "standardized_name"],
            ascending=[True, False, False, True],
        )
    )
    counts.insert(1, "entity_type", entity_type)
    counts[rank_column] = counts.groupby("research_field").cumcount() + 1
    return counts.reset_index(drop=True)


field_resource_counts = count_sources_by_field(
    source_mentions, all_field_memberships, RESOURCE, "rank_in_field"
)
field_holder_counts = count_sources_by_field(
    source_mentions, all_field_memberships, HOLDER, "rank_in_field"
)
primary_field_resource_counts = count_sources_by_field(
    source_mentions, primary_field_memberships, RESOURCE, "rank_in_primary_field"
)
primary_field_holder_counts = count_sources_by_field(
    source_mentions, primary_field_memberships, HOLDER, "rank_in_primary_field"
)

FIELD_OUTPUTS = {
    "research_field_data_resource_product_counts.csv": field_resource_counts,
    "research_field_data_holder_counts.csv": field_holder_counts,
    "primary_research_field_data_resource_product_counts.csv": primary_field_resource_counts,
    "primary_research_field_data_holder_counts.csv": primary_field_holder_counts,
}
for filename, table in FIELD_OUTPUTS.items():
    table.to_csv(OUTPUT_DIR / filename, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUTPUT_DIR / filename}")

display(field_resource_counts.query("rank_in_field <= @TOP_N_BY_FIELD").head(50))

### Plot field-specific rankings

Each panel represents one broad field. These plots show both the overlapping all-membership definition and the mutually exclusive primary-field definition.

In [ ]:
def plot_field_counts(
    counts, rank_column, title, filename_stem, color, orientation="horizontal",
    show_title=True, save_pdf=False, pdf_filename=None,
):
    fields = [field for field in FIELD_ORDER if field in set(counts["research_field"])]
    ncols = 2
    nrows = math.ceil(len(fields) / ncols)
    row_height = 5.8 if orientation == "vertical" else 4.3
    figure, axes = plt.subplots(
        nrows, ncols, figsize=(15, row_height * nrows), squeeze=False
    )

    for axis, field in zip(axes.ravel(), fields):
        plot_data = (
            counts.loc[
                counts["research_field"].eq(field) & counts[rank_column].le(TOP_N_BY_FIELD)
            ]
            .sort_values(
                "unique_papers", ascending=orientation == "horizontal"
            )
            .copy()
        )
        if orientation == "vertical":
            plot_data["label"] = plot_data["standardized_name"].map(
                lambda value: "\n".join(wrap(value, width=18))
            )
            axis.bar(plot_data["label"], plot_data["unique_papers"], color=color)
            axis.set(
                title=FIELD_DISPLAY.get(field, field),
                xlabel="",
                ylabel="Unique papers",
            )
            axis.grid(axis="y", alpha=0.25)
            axis.tick_params(axis="x", labelrotation=55, labelsize=9)
            for tick_label in axis.get_xticklabels():
                tick_label.set_ha("right")
                tick_label.set_rotation_mode("anchor")
        else:
            plot_data["label"] = plot_data["standardized_name"].map(
                lambda value: "\n".join(wrap(value, width=28))
            )
            axis.barh(
                plot_data["label"], plot_data["unique_papers"], color=color
            )
            axis.set(
                title=FIELD_DISPLAY.get(field, field),
                xlabel="Unique papers",
                ylabel="",
            )
            axis.grid(axis="x", alpha=0.25)
        axis.spines[["top", "right"]].set_visible(False)

    for axis in axes.ravel()[len(fields):]:
        axis.axis("off")

    if show_title:
        figure.suptitle(title, fontsize=14)
    figure.tight_layout(pad=2.0)
    save_matplotlib_figure(figure, filename_stem)
    if save_pdf:
        figure.savefig(
            FIGURE_DIR / f"{pdf_filename or filename_stem}.pdf",
            format="pdf",
            bbox_inches="tight",
        )
    plt.show()


plot_field_counts(
    field_resource_counts, "rank_in_field",
    f"Top {TOP_N_BY_FIELD} resources by field: all memberships",
    "research_field_top_data_resource_product_counts", "#2f6f9f",
)
plot_field_counts(
    field_holder_counts, "rank_in_field",
    f"Top {TOP_N_BY_FIELD} holders by field: all memberships",
    "research_field_top_data_holder_counts", "#7a4f9a",
)
plot_field_counts(
    primary_field_resource_counts, "rank_in_primary_field",
    f"Top {TOP_N_BY_FIELD} resources by primary field",
    "primary_research_field_top_data_resource_product_counts", "#2f6f9f",
)
plot_field_counts(
    primary_field_holder_counts, "rank_in_primary_field",
    f"Top {TOP_N_BY_FIELD} holders by primary field",
    "primary_research_field_top_data_holder_counts", "#2f6f9f",
    orientation="vertical",
    show_title=False,
    save_pdf=True,
    pdf_filename="SFig1",
)

## 10. Holder-use distributions

Panels b and c use the same counting unit: one deduplicated paper–holder pair is one data-use instance. Their scopes differ:

- panel b groups all paper–holder pairs in the retained corpus by data holder type;
- panel c uses an inner join and groups the pairs from papers with a mapped primary research field.

This keeps panel b unchanged while making panel c consistent with the primary-field holder bar plots. Using holder mentions only avoids double-counting the same conceptual relationship once as a resource row and again as a holder row.

In [ ]:
# Panel b remains unchanged and uses every deduplicated paper–holder pair.
holder_type_distribution = (
    holder_mentions.assign(
        standardized_data_holder_type=lambda frame: frame[
            "standardized_data_holder_type"
        ].fillna("not specified")
    )
    .groupby("standardized_data_holder_type", as_index=False)
    .agg(
        holder_mentions=("paper_id", "size"),
        unique_papers=("paper_id", "nunique"),
    )
    .sort_values("holder_mentions", ascending=False)
)
holder_type_distribution["share_of_holder_mentions"] = (
    holder_type_distribution["holder_mentions"]
    / holder_type_distribution["holder_mentions"].sum()
)

# Keep only paper–holder use instances from papers with a primary research field.
# This is the same inner join used by the primary-field holder bar plots.
mapped_holder_mentions = holder_mentions.merge(
    primary_field_memberships,
    on="paper_id",
    how="inner",
    validate="many_to_one",
)

# Panel c groups the mapped paper–holder use instances by primary field.
field_distribution = (
    mapped_holder_mentions
    .groupby("research_field", as_index=False)
    .agg(
        holder_mentions=("paper_id", "size"),
        unique_papers=("paper_id", "nunique"),
    )
    .sort_values("holder_mentions", ascending=False)
)
field_distribution["share_of_holder_mentions"] = (
    field_distribution["holder_mentions"]
    / field_distribution["holder_mentions"].sum()
)

# Confirm the separate full-corpus and mapped-paper totals.
assert holder_type_distribution["holder_mentions"].sum() == len(holder_mentions)
assert field_distribution["holder_mentions"].sum() == len(mapped_holder_mentions)

holder_type_distribution.to_csv(
    OUTPUT_DIR / "full_corpus_data_holder_type_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)
field_distribution.to_csv(
    OUTPUT_DIR / "full_corpus_primary_research_field_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

FIELD_COLORS = {
    "Ecology, Biodiversity & Conservation": "#009e73",
    "Earth, Climate, Ocean & Water": "#56b4e9",
    "Agriculture, Forestry & Food": "#e69f00",
    "Environmental Health, Pollution & Risk": "#d55e00",
    "Social, Economic & Policy": "#cc79a7",
}

def plot_distribution_bar(
    table, label, value, title, filename_stem, colors, label_width, figure_width,
    display_map=None,
):
    """Draw a standalone vertical count bar chart with percentage labels."""
    plot_data = table.sort_values(value, ascending=False)
    total = plot_data[value].sum()
    x_positions = np.arange(len(plot_data))
    figure, axis = plt.subplots(figsize=(figure_width, 6.5))
    bars = axis.bar(
        x_positions,
        plot_data[value],
        color=[colors.get(item, "#bdbdbd") for item in plot_data[label]],
        width=0.68,
    )
    axis.bar_label(
        bars,
        labels=[
            f"{row[value]:,}\n({row[value] / total:.1%})"
            for _, row in plot_data.iterrows()
        ],
        padding=4,
        fontsize=10,
    )
    display_map = display_map or {}
    display_labels = [display_map.get(item, item) for item in plot_data[label]]
    axis.set_xticks(
        x_positions,
        ["\n".join(wrap(str(item), width=label_width)) for item in display_labels],
    )
    axis.set_ylabel("Data use instances")
    axis.set_xlabel("")
    axis.set_title(title, loc="left")
    axis.set_ylim(0, plot_data[value].max() * 1.18)
    axis.grid(axis="y", alpha=0.22)
    axis.set_axisbelow(True)
    axis.spines[["top", "right"]].set_visible(False)
    figure.subplots_adjust(left=0.10, right=0.98, top=0.88, bottom=0.25)
    save_matplotlib_figure(figure, filename_stem)
    plt.show()


plot_distribution_bar(
    holder_type_distribution, "standardized_data_holder_type", "holder_mentions",
    "Distribution of holder types across data use instances",
    "full_corpus_data_holder_type_distribution_bar", HOLDER_TYPE_COLORS,
    label_width=15, figure_width=10.5, display_map=HOLDER_TYPE_DISPLAY,
)
plot_distribution_bar(
    field_distribution, "research_field", "holder_mentions",
    "Distribution of research fields across data use instances",
    "full_corpus_primary_research_field_distribution_bar", FIELD_COLORS,
    label_width=17, figure_width=12.5,
)

display(holder_type_distribution)
display(field_distribution)

In [ ]:
# Total number of unique paper–holder use instances.
total_holder_use_instances = holder_type_distribution["holder_mentions"].sum()

print(f"Panel b — all paper–holder use instances: {total_holder_use_instances:,}")

# Confirm that this equals the number of rows in the deduplicated holder table.
assert total_holder_use_instances == len(holder_mentions)

# Panel c contains only use instances from papers with a primary field.
mapped_holder_use_instances = field_distribution["holder_mentions"].sum()
print(f"Panel c — field-mappable paper–holder use instances: {mapped_holder_use_instances:,}")
assert mapped_holder_use_instances == len(mapped_holder_mentions)

### Standalone trial: assigned papers versus observed data-use instances

This additional analysis does not replace or modify the existing figures. For each primary research field, paired bars compare the number of assigned papers with the number of observed data-use instances. A data-use instance is one unique paper–data-holder pair. The annotation reports data-use instances per assigned paper, with every assigned paper retained in the denominator, including papers with zero matched holders.

In [ ]:
# Count the number of unique data holders used by each paper.
paper_holder_use_counts = (
    holder_mentions
    .groupby("paper_id", as_index=False)
    .agg(unique_data_holders=("standardized_name", "nunique"))
)

# Start with every field-mappable paper so papers with no matched holder count as zero.
field_holder_uses_per_paper = primary_field_memberships.merge(
    paper_holder_use_counts,
    on="paper_id",
    how="left",
    validate="one_to_one",
)
field_holder_uses_per_paper["unique_data_holders"] = (
    field_holder_uses_per_paper["unique_data_holders"].fillna(0).astype(int)
)

# Summarize the number of papers, holder uses, and use intensity in each field.
field_holder_use_intensity = (
    field_holder_uses_per_paper
    .groupby("research_field", as_index=False)
    .agg(
        papers=("paper_id", "nunique"),
        total_holder_uses=("unique_data_holders", "sum"),
        mean_holders_per_paper=("unique_data_holders", "mean"),
        median_holders_per_paper=("unique_data_holders", "median"),
    )
)

# Compare each field with the overall mean across all field-mappable papers.
overall_mean_holders_per_paper = (
    field_holder_uses_per_paper["unique_data_holders"].mean()
)
field_holder_use_intensity["relative_use_intensity"] = (
    field_holder_use_intensity["mean_holders_per_paper"]
    / overall_mean_holders_per_paper
)
field_holder_use_intensity = field_holder_use_intensity.sort_values(
    "mean_holders_per_paper", ascending=True
)

# Create and save the compact comparison table underlying the chart.
field_paper_use_comparison = (
    field_holder_use_intensity
    .rename(columns={
        "papers": "assigned_papers",
        "total_holder_uses": "data_use_instances",
        "mean_holders_per_paper": "data_use_instances_per_paper",
    })[
        [
            "research_field",
            "assigned_papers",
            "data_use_instances",
            "data_use_instances_per_paper",
        ]
    ]
    .sort_values("data_use_instances_per_paper", ascending=False)
    .reset_index(drop=True)
)
field_paper_use_comparison.to_csv(
    OUTPUT_DIR / "primary_research_field_papers_vs_data_use_instances.csv",
    index=False,
    encoding="utf-8-sig",
)

# Plot low-to-high so the largest use-per-paper ratio appears at the top.
plot_data = field_paper_use_comparison.sort_values(
    "data_use_instances_per_paper", ascending=True
).reset_index(drop=True)
y_positions = np.arange(len(plot_data))
bar_height = 0.34
figure, axis = plt.subplots(figsize=(13, 7.2))
paper_bars = axis.barh(
    y_positions - bar_height / 2,
    plot_data["assigned_papers"],
    height=bar_height,
    color="#c9c9c9",
    label="Assigned papers",
)
use_bars = axis.barh(
    y_positions + bar_height / 2,
    plot_data["data_use_instances"],
    height=bar_height,
    color=[FIELD_COLORS.get(field, "#777777") for field in plot_data["research_field"]],
    edgecolor=[
        "#222222" if field == "Genetics & Molecular Biology" else "none"
        for field in plot_data["research_field"]
    ],
    linewidth=1.2,
    label="Observed data-use instances",
)

# Show the two counts and a separate use-per-paper annotation for each field.
axis.bar_label(
    paper_bars,
    labels=[f"{value:,.0f}" for value in plot_data["assigned_papers"]],
    padding=3,
    fontsize=9,
)
axis.bar_label(
    use_bars,
    labels=[f"{value:,.0f}" for value in plot_data["data_use_instances"]],
    padding=3,
    fontsize=9,
)
count_maximum = plot_data[["assigned_papers", "data_use_instances"]].to_numpy().max()
ratio_label_x = count_maximum * 1.16
for y_position, (_, row) in zip(y_positions, plot_data.iterrows()):
    is_genetics = row["research_field"] == "Genetics & Molecular Biology"
    axis.text(
        ratio_label_x,
        y_position,
        f"{row['data_use_instances_per_paper']:.2f} per paper",
        va="center",
        ha="left",
        fontsize=10,
        fontweight="bold" if is_genetics else "normal",
        color="#222222",
    )

axis.set_yticks(y_positions, plot_data["research_field"])
axis.set_xlabel("Count")
axis.set_ylabel("")
axis.set_title("Papers and data-use instances by primary research field", loc="left")
axis.grid(axis="x", alpha=0.22)
axis.set_axisbelow(True)
axis.spines[["top", "right"]].set_visible(False)
axis.set_xlim(0, count_maximum * 1.43)
axis.legend(
    handles=[
        Patch(facecolor="#c9c9c9", label="Assigned papers"),
        Patch(facecolor="#666666", label="Observed data-use instances (field color)"),
    ],
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.11),
    ncol=2,
)
figure.text(
    0.01,
    0.015,
    "Note: one data-use instance is one unique paper–data-holder pair; assigned papers with no matched holder remain in the denominator.",
    fontsize=9,
    color="#444444",
)
figure.subplots_adjust(left=0.35, right=0.98, top=0.88, bottom=0.20)
save_matplotlib_figure(
    figure, "primary_research_field_papers_vs_data_use_instances"
)
plt.show()

display(field_paper_use_comparison)

## 11. Sensitivity check: treat EU institutions as national public

This calculation uses the `EU` indicator in the `unique_institutions` sheet. EU-flagged holders currently classified as `international public` are reassigned to `national public`; all other holder types remain unchanged. Shares use the same denominator as the main holder-type distribution: all deduplicated paper–holder pairs in `holder_mentions`.

In [ ]:
# Load the holder names and EU indicator from the supplementary workbook.
eu_holder_flags = pd.read_excel(
    SUPPLEMENTARY_FILE,
    sheet_name="unique_institutions",
    usecols=["data_holder", "EU"],
)

# Clean the holder names and retain only rows marked as EU institutions.
eu_holder_flags["data_holder"] = (
    eu_holder_flags["data_holder"].astype("string").str.strip()
)
eu_holder_flags["EU"] = pd.to_numeric(
    eu_holder_flags["EU"], errors="coerce"
).fillna(0)
eu_holder_names = set(
    eu_holder_flags.loc[eu_holder_flags["EU"].eq(1), "data_holder"].dropna()
)

# Copy the deduplicated paper–holder table used in the main distribution.
holder_mentions_eu_adjusted = holder_mentions.copy()
holder_mentions_eu_adjusted["adjusted_data_holder_type"] = (
    holder_mentions_eu_adjusted["standardized_data_holder_type"]
    .fillna("not specified")
)

# Reclassify EU institutions only when their current type is international public.
eu_international_mask = (
    holder_mentions_eu_adjusted["standardized_name"].isin(eu_holder_names)
    & holder_mentions_eu_adjusted["adjusted_data_holder_type"].eq("international public")
)
holder_mentions_eu_adjusted.loc[
    eu_international_mask, "adjusted_data_holder_type"
] = "national public"

# Count paper–holder pairs by the adjusted type and calculate their shares.
adjusted_holder_type_distribution = (
    holder_mentions_eu_adjusted
    .groupby("adjusted_data_holder_type", as_index=False)
    .agg(
        holder_mentions=("paper_id", "size"),
        unique_papers=("paper_id", "nunique"),
    )
    .sort_values("holder_mentions", ascending=False)
)
adjusted_holder_type_distribution["share_of_holder_mentions"] = (
    adjusted_holder_type_distribution["holder_mentions"]
    / adjusted_holder_type_distribution["holder_mentions"].sum()
)

# Save the EU-adjusted distribution used by Figure 1 panel b.
adjusted_holder_type_distribution_for_figure = (
    adjusted_holder_type_distribution.rename(columns={
        "adjusted_data_holder_type": "standardized_data_holder_type"
    })
)
adjusted_holder_type_distribution_for_figure.to_csv(
    OUTPUT_DIR / "full_corpus_data_holder_type_distribution_eu_adjusted.csv",
    index=False,
    encoding="utf-8-sig",
)

# Show how many paper–holder pairs moved and display the adjusted shares.
print(f"EU paper–holder pairs moved to national public: {eu_international_mask.sum():,}")
display(adjusted_holder_type_distribution_for_figure)

## 12. Combined overview figure

The final figure places the holder–resource treemap above the two distributions. It is generated directly from the in-memory tables and does not depend on intermediate figure files.

The interactive HTML is useful for checking labels and hover details. The export cell writes an untouched editable SVG (_editable.svg), a sanitized SVG for insertion into Microsoft Word, and a high-resolution PNG fallback. The canonical paper_figure_data_sources_overview.svg is the Word-compatible version.

In [ ]:
overview = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "treemap", "colspan": 2}, None],
        [{"type": "xy"}, {"type": "xy"}],
    ],
    subplot_titles=("a", "b", "c"),
    row_heights=[0.62, 0.38],
    vertical_spacing=0.10,
    horizontal_spacing=0.09,
)

overview.add_trace(treemap.data[0], row=1, col=1)
overview.data[0].domain = dict(x=[0, 1], y=[0.445, 1])

# Panel b: data-use instances by holder type.
holder_bar_data = holder_type_distribution.sort_values(
    "holder_mentions", ascending=False
).reset_index(drop=True)
holder_bar_labels = [
    "<br>".join(wrap(
        HOLDER_TYPE_DISPLAY.get(holder_type, holder_type), width=15
    ))
    for holder_type in holder_bar_data["standardized_data_holder_type"]
]
overview.add_trace(
    go.Bar(
        x=holder_bar_labels,
        y=holder_bar_data["holder_mentions"],
        marker_color=[
            HOLDER_TYPE_COLORS.get(holder_type, "#bdbdbd")
            for holder_type in holder_bar_data["standardized_data_holder_type"]
        ],
        text=[
            f"{row['holder_mentions']:,}<br>({row['share_of_holder_mentions']:.1%})"
            for _, row in holder_bar_data.iterrows()
        ],
        textposition="outside",
        textfont=dict(size=13),
        cliponaxis=False,
        width=0.70,
        showlegend=False,
        hovertemplate="%{x}<br>%{y:,} data use instances<extra></extra>",
    ),
    row=2, col=1,
)

# Panel c: data-use instances by primary research field.
field_bar_data = field_distribution.sort_values(
    "holder_mentions", ascending=False
).reset_index(drop=True)
field_bar_labels = [
    "<br>".join(wrap(field, width=17))
    for field in field_bar_data["research_field"]
]
overview.add_trace(
    go.Bar(
        x=field_bar_labels,
        y=field_bar_data["holder_mentions"],
        marker_color=[
            FIELD_COLORS.get(field, "#bdbdbd")
            for field in field_bar_data["research_field"]
        ],
        text=[
            f"{row['holder_mentions']:,}<br>({row['share_of_holder_mentions']:.1%})"
            for _, row in field_bar_data.iterrows()
        ],
        textposition="outside",
        textfont=dict(size=13),
        cliponaxis=False,
        width=0.70,
        showlegend=False,
        hovertemplate="%{x}<br>%{y:,} data use instances<extra></extra>",
    ),
    row=2, col=2,
)

# Add a horizontal legend beneath panel a for holder types represented in the treemap.
treemap_holder_types = [
    holder_type
    for holder_type in HOLDER_TYPE_COLORS
    if holder_type in set(treemap_counts["standardized_data_holder_type"])
]
for holder_type in treemap_holder_types:
    overview.add_trace(
        go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(
                size=15,
                color=HOLDER_TYPE_COLORS[holder_type],
                symbol="square",
            ),
            name=HOLDER_TYPE_DISPLAY.get(holder_type, holder_type),
            showlegend=True,
            hoverinfo="skip",
        ),
        row=2, col=1,
    )

# Fix the lower-panel domains to leave room for headings and the panel-a legend.
overview.update_xaxes(domain=[0.035, 0.47], row=2, col=1)
overview.update_xaxes(domain=[0.56, 0.995], row=2, col=2)
overview.update_yaxes(domain=[0.075, 0.34], row=2, col=1)
overview.update_yaxes(domain=[0.075, 0.34], row=2, col=2)
overview.update_yaxes(
    title_text="Data use instances",
    range=[0, holder_bar_data["holder_mentions"].max() * 1.20],
    tickformat=",d",
    showgrid=True, gridcolor="#e5e5e5", zeroline=False,
    tickfont=dict(size=15), title_font=dict(size=15),
    title_standoff=8, automargin=True,
    row=2, col=1,
)
overview.update_yaxes(
    title_text="Data use instances",
    range=[0, field_bar_data["holder_mentions"].max() * 1.20],
    tickformat=",d",
    showgrid=True, gridcolor="#e5e5e5", zeroline=False,
    tickfont=dict(size=15), title_font=dict(size=15),
    title_standoff=8, automargin=True,
    row=2, col=2,
)
overview.update_xaxes(
    tickfont=dict(size=15), showgrid=False, automargin=True,
    row=2, col=1,
)
overview.update_xaxes(
    tickfont=dict(size=15), showgrid=False, automargin=True,
    row=2, col=2,
)

overview.update_layout(
    width=1500,
    height=1600,
    margin=dict(t=60, l=65, r=20, b=95),
    paper_bgcolor="white",
    plot_bgcolor="white",
    bargap=0.24,
    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.37, y=0.423, xanchor="left", yanchor="middle",
        font=dict(size=15),
        itemsizing="constant",
        traceorder="normal",
    ),
    font=dict(family="Arial, Helvetica, sans-serif", color="#111111"),
)

# Place panel letters and headings consistently above each plot.
panel_label_positions = [(0.0, 1.0, 8), (0.0, 0.37, 0), (0.525, 0.37, 0)]
for annotation, (x_position, y_position, y_shift) in zip(
    overview.layout.annotations, panel_label_positions
):
    annotation.update(
        x=x_position, y=y_position, xanchor="left", yanchor="bottom",
        yshift=y_shift,
        font=dict(size=24, family="Arial Black, Arial"),
    )

panel_headings = [
    (0.035, 1.0, "15 most frequently used data holders and their resources", 8),
    (0.035, 0.37, "Distribution of holder types across data use instances", 0),
    (0.56, 0.37, "Distribution of research fields across data use instances", 0),
]
for x_position, y_position, heading, y_shift in panel_headings:
    overview.add_annotation(
        x=x_position, y=y_position, xref="paper", yref="paper",
        text=heading, showarrow=False, xanchor="left", yanchor="bottom",
        yshift=y_shift,
        font=dict(size=18, family="Arial, Helvetica, sans-serif", color="#111111"),
    )

overview.add_annotation(
    x=0.35, y=0.423, xref="paper", yref="paper",
    text="Data holder type", showarrow=False,
    xanchor="right", yanchor="middle",
    font=dict(size=15, family="Arial, Helvetica, sans-serif", color="#333333"),
)

overview.write_html(FIGURE_DIR / "paper_figure_data_sources_overview.html")
save_plotly_figure_for_office(
    overview,
    "paper_figure_data_sources_overview",
    width=1500,
    height=1600,
)
overview.show()

## 13. Final checks

This last cell summarizes the main row counts and checks that the canonical holder categories from the final workbook were preserved. Unexpected missing metadata is printed for review rather than silently recoded.

In [ ]:
expected_holder_types = {
    "national public", "international public", "research",
    "private", "ngo-led", "hybrid",
}
observed_holder_types = set(
    holder_mentions["standardized_data_holder_type"].dropna().unique()
)

print("Analysis complete.")
print(f"Retained papers: {len(analysis_papers):,}")
print(f"Paper–resource pairs: {len(resource_mentions):,}")
print(f"Paper–holder pairs: {len(holder_mentions):,}")
print(f"Unique resources: {resource_mentions['standardized_name'].nunique():,}")
print(f"Unique holders: {holder_mentions['standardized_name'].nunique():,}")
print(f"Papers assigned a primary broad field: {primary_field_memberships['paper_id'].nunique():,}")

unexpected_types = observed_holder_types - expected_holder_types
if unexpected_types:
    print(f"Review unexpected holder types: {sorted(unexpected_types)}")

missing_resource_metadata = resource_mentions["standardized_data_resource_name"].isna().sum()
missing_holder_metadata = holder_mentions["standardized_data_holder_name"].isna().sum()
print(f"Resource mentions without matching metadata: {missing_resource_metadata:,}")
print(f"Holder mentions without matching visualization metadata: {missing_holder_metadata:,}")

In [ ]:
# Rebuild the final combined figure as a compact Matplotlib SVG.
# Tiny unlabeled treemap boxes are consolidated into one 'Other resources'
# box per holder, avoiding the SVG complexity that Word can drop on PDF export.
from runpy import run_path

figure_builder = Path.cwd() / 'build_simplified_figure1.py'
if not figure_builder.exists():
    figure_builder = (
        Path.cwd() / 'code' / 'text_mining' / 'build_simplified_figure1.py'
    )
run_path(figure_builder, run_name='__main__')